# Transformer Baselines

This notebook runs Hugging Face transformer comparisons and creates analysis tables:

- `total_results`: all evaluated transformer runs.
- `report_results`: the best validation result per model.
- `epoch_history`: per-epoch validation metrics for training curves.

The task predicts sentiment labels `0..4`, with validation score `1 - MAE / 4`.

In [ ]:
from datetime import datetime
from pathlib import Path
import subprocess
import sys

import pandas as pd
import torch

EXPERIMENT_KIND = "TRANSFORMER_BASELINES"
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
EXPERIMENT_DIR = Path("experiments/transformers") / f"{RUN_ID}_{EXPERIMENT_KIND}"
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = Path("data/train.csv")
VALIDATION_SIZE = 0.1
RANDOM_STATE = 42
MAX_LENGTH = 256
EPOCHS = 3
BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
USE_FP16 = True

EXPERIMENT_DIR


In [ ]:
models = [
    # {"model_name": "distilbert-base-uncased", "variant": "fine_tuned"},  # completed
    # {"model_name": "bert-base-uncased", "variant": "fine_tuned"},  # completed
    # {"model_name": "roberta-base", "variant": "fine_tuned"},  # completed
    {"model_name": "nlptown/bert-base-multilingual-uncased-sentiment", "variant": "fine_tuned"},
    {"model_name": "nlptown/bert-base-multilingual-uncased-sentiment", "variant": "eval_only"},
]

models


## Run Experiments

In [ ]:
completed_runs = []

for spec in models:
    model_name = spec["model_name"]
    variant = spec["variant"]
    run_name = f"{model_name.replace('/', '__')}__{variant}"
    run_dir = EXPERIMENT_DIR / run_name
    run_dir.mkdir(parents=True, exist_ok=True)

    cmd = [
        sys.executable,
        "-m",
        "baselines.train_review_model",
        "--model-name",
        model_name,
        "--experiment-name",
        f"transformer__{run_name}",
        "--train-path",
        str(TRAIN_PATH),
        "--output-dir",
        str(run_dir),
        "--validation-size",
        str(VALIDATION_SIZE),
        "--random-state",
        str(RANDOM_STATE),
        "--max-length",
        str(MAX_LENGTH),
        "--epochs",
        str(EPOCHS),
        "--batch-size",
        str(BATCH_SIZE),
        "--eval-batch-size",
        str(EVAL_BATCH_SIZE),
        "--learning-rate",
        str(LEARNING_RATE),
        "--weight-decay",
        str(WEIGHT_DECAY),
    ]
    if variant == "eval_only":
        cmd.append("--eval-only")
    if USE_FP16:
        cmd.append("--fp16")

    print(" ".join(cmd))
    subprocess.run(cmd, check=True)
    completed_runs.append({**spec, "run_name": run_name, "run_dir": run_dir})

completed_runs


## Total Analysis

In [ ]:
result_frames = []
for run in completed_runs:
    result_frames.append(pd.read_csv(run["run_dir"] / "transformer_results.csv"))

results = pd.concat(result_frames, ignore_index=True)
total_results = results.sort_values(
    ["status", "cil_score"], ascending=[False, False]
).reset_index(drop=True)
total_results.to_csv(EXPERIMENT_DIR / "total_analysis.csv", index=False)
total_results

## Report Analysis

In [ ]:
ok_results = results[results["status"] == "ok"].copy()
report_results = (
    ok_results.sort_values("cil_score", ascending=False)
    .groupby(["model", "variant"], as_index=False)
    .first()
    .sort_values("cil_score", ascending=False)
    .reset_index(drop=True)
)
report_results.to_csv(EXPERIMENT_DIR / "report_analysis.csv", index=False)
report_results

## Epoch History

In [ ]:
history_frames = []
for run in completed_runs:
    history_frames.append(pd.read_csv(run["run_dir"] / "transformer_epoch_history.csv"))

epoch_history = pd.concat(history_frames, ignore_index=True)
epoch_history = epoch_history.sort_values(["experiment", "epoch"]).reset_index(drop=True)
epoch_history.to_csv(EXPERIMENT_DIR / "epoch_analysis.csv", index=False)
epoch_history

## Analysis Artifacts


In [ ]:
analysis_dir = EXPERIMENT_DIR / "analysis"
analysis_dir.mkdir(parents=True, exist_ok=True)

transformer_epoch_plot = analysis_dir / "transformer_epoch_cil_scores.pdf"
subprocess.run(
    [
        sys.executable,
        "-m",
        "baselines.analysis.plot_epoch_scores",
        "--input",
        str(EXPERIMENT_DIR / "epoch_analysis.csv"),
        "--output-dir",
        str(analysis_dir),
        "--prefix",
        "transformer_epoch_cil_scores",
        "--title",
        "Transformer epoch analysis",
        "--caption",
        "Validation CIL-score by epoch for transformer baselines.",
        "--label",
        "fig:transformer-epoch-cil-scores",
    ],
    check=True,
)

transformer_epoch_plot
